In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive"
DRIVE_DIR = ROOT + "/files"
DATA_DIR = DRIVE_DIR + "/data"

for name in ("sources.yaml", "fetch_raw.py"):
    assert os.path.exists(f"{DRIVE_DIR}/{name}"), f"{name} absent de {DRIVE_DIR}"
    shutil.copy(f"{DRIVE_DIR}/{name}", ".")
print("Scripts copies depuis", DRIVE_DIR, "| donnees dans", DATA_DIR)

Mounted at /content/drive
Scripts copies depuis /content/drive/MyDrive/files | donnees dans /content/drive/MyDrive/files/data


In [ ]:
%%writefile scan_pfam_domains.py
"""scan_pfam_domains.py : scanne un proteome contre les modeles Pfam-A (deja presses
avec hmmpress) et produit une table proteine <-> domaine. Decoupe le proteome en
lots pour pouvoir reprendre un scan interrompu (deconnexion Colab, changement de
runtime, fin de session...).
"""
import argparse
import csv
import gzip
import subprocess
import time
from pathlib import Path


def iter_fasta_records(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8", errors="replace") as f:
        header, seq_lines = None, []
        for line in f:
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_lines)
                header, seq_lines = line.rstrip("\n"), []
            else:
                seq_lines.append(line)
        if header is not None:
            yield header, "".join(seq_lines)


def split_into_chunks(protein_path, chunks_dir, chunk_size):
    chunks_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(chunks_dir.glob("chunk_*.faa"))
    if existing:
        print(f"{len(existing)} lot(s) deja decoupe(s) dans {chunks_dir} (reutilises, pas re-decoupes).")
        return existing

    chunk_paths, buf, idx, count = [], [], 0, 0

    def flush():
        nonlocal idx, buf
        if not buf:
            return
        p = chunks_dir / f"chunk_{idx:05d}.faa"
        p.write_text("".join(buf), encoding="utf-8")
        chunk_paths.append(p)
        idx += 1
        buf = []

    for header, seq in iter_fasta_records(protein_path):
        buf.append(header + "\n" + seq)
        count += 1
        if count % chunk_size == 0:
            flush()
    flush()
    print(f"{count} proteines decoupees en {len(chunk_paths)} lot(s) de {chunk_size} dans {chunks_dir}")
    return chunk_paths


def is_domtbl_complete(path):
    """Un domtblout est complet seulement si hmmscan a fini normalement :
    HMMER ecrit une ligne '# [ok]' a la toute fin. Un run interrompu
    (deconnexion, changement de runtime) laisse un fichier non vide mais
    tronque, sans cette ligne -- un simple test de taille ne suffit pas."""
    if not path.exists() or path.stat().st_size == 0:
        return False
    with open(path, "rb") as f:
        f.seek(max(0, path.stat().st_size - 200))
        tail = f.read().decode("utf-8", errors="replace")
    return "# [ok]" in tail


def run_hmmscan(pfam_hmm, protein_fasta, domtbl_out, cpu):
    cmd = ["hmmscan", "--cut_ga", "--cpu", str(cpu),
           "--domtblout", str(domtbl_out), "-o", "/dev/null",
           str(pfam_hmm), str(protein_fasta)]
    t0 = time.time()
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print("  --cut_ga a echoue (seuil GA absent sur un modele Pfam) -- repli sur -E 1e-5 pour ce lot")
        cmd_fallback = ["hmmscan", "-E", "1e-5", "--cpu", str(cpu),
                         "--domtblout", str(domtbl_out), "-o", "/dev/null",
                         str(pfam_hmm), str(protein_fasta)]
        subprocess.run(cmd_fallback, check=True)
    return time.time() - t0


def parse_domtblout(path, species, assembly):
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            cols = line.split()
            if len(cols) < 19:
                continue
            rows.append({
                "species": species, "assembly": assembly,
                "protein_accession": cols[3], "pfam_name": cols[0],
                "pfam_accession": cols[1], "evalue": cols[6], "score": cols[7],
                "ali_from": cols[17], "ali_to": cols[18],
            })
    return rows


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--pfam-hmm", required=True)
    ap.add_argument("--proteins", required=True)
    ap.add_argument("--species", required=True)
    ap.add_argument("--assembly", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--cpu", type=int, default=4)
    ap.add_argument("--chunk-size", type=int, default=2000)
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    chunks_dir = out_dir / "chunks"
    domtbl_dir = out_dir / "domtblout"
    domtbl_dir.mkdir(parents=True, exist_ok=True)

    if args.limit:
        chunks_dir.mkdir(parents=True, exist_ok=True)
        test_chunk = chunks_dir / "chunk_test.faa"
        with open(test_chunk, "w", encoding="utf-8") as f:
            for i, (header, seq) in enumerate(iter_fasta_records(args.proteins)):
                if i >= args.limit:
                    break
                f.write(header + "\n" + seq)
        chunk_paths = [test_chunk]
    else:
        chunk_paths = split_into_chunks(Path(args.proteins), chunks_dir, args.chunk_size)

    done, skipped = 0, 0
    for chunk_path in chunk_paths:
        domtbl_path = domtbl_dir / (chunk_path.stem + ".domtblout.txt")
        if is_domtbl_complete(domtbl_path):
            skipped += 1
            continue
        print(f"$ hmmscan sur {chunk_path.name} ...", flush=True)
        elapsed = run_hmmscan(args.pfam_hmm, chunk_path, domtbl_path, args.cpu)
        done += 1
        print(f"  termine en {elapsed:.0f} s")

    print(f"\n{done} lot(s) scanne(s) cette execution, {skipped} deja fait(s) (sautes).")

    all_rows = []
    available = sorted(domtbl_dir.glob("*.domtblout.txt"))
    complete_available = [p for p in available if is_domtbl_complete(p)]
    for p in complete_available:
        all_rows.extend(parse_domtblout(p, args.species, args.assembly))
    out_csv = out_dir / f"pfam_domains_{args.species}.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["species", "assembly", "protein_accession", "pfam_name",
                                           "pfam_accession", "evalue", "score", "ali_from", "ali_to"])
        w.writeheader()
        w.writerows(all_rows)

    n_proteins = len({r["protein_accession"] for r in all_rows})
    complet = len(complete_available) >= len(chunk_paths)
    print(f"{len(complete_available)}/{len(chunk_paths)} lot(s) complet(s) et disponible(s) au total "
          f"({'COMPLET' if complet else 'PARTIEL -- relance la meme commande pour continuer'})")
    print(f"{len(all_rows)} hits, {n_proteins} proteines avec au moins un domaine (sur les lots complets disponibles)")
    print(f"-> {out_csv}")


if __name__ == "__main__":
    main()

Writing scan_pfam_domains.py


In [ ]:
shutil.copy("scan_pfam_domains.py", f"{DRIVE_DIR}/scan_pfam_domains.py")

'/content/drive/MyDrive/files/scan_pfam_domains.py'

In [ ]:
# Installer hmmer d'abord
!apt-get -qq install hmmer

Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-6_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-6) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.4+dfsg-2_amd64.deb ...
Unpacking hmmer (3.4+dfsg-2) ...
Setting up libdivsufsort3:amd64 (2.0.1-6) ...
Setting up hmmer (3.4+dfsg-2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/li

In [ ]:
import os

os.makedirs("/content/pfam", exist_ok=True)
source_hmm = f"{DATA_DIR}/raw/shared/interpro_pfam/rolling_2026-10-12/Pfam-A.hmm"
hmm_path = "/content/pfam/Pfam-A.hmm"
names_file = "/content/pfam/keep_names.txt"

if not os.path.exists(hmm_path) and os.path.exists(source_hmm):
    os.symlink(source_hmm, hmm_path)

with open(hmm_path, encoding="utf-8", errors="replace") as f, open(names_file, "w") as out:
    for line in f:
        if line.startswith("NAME  "):
            name = line.split()[1]
            if name != "UPF0236":
                out.write(name + "\n")

!hmmfetch --index "{hmm_path}"
!hmmfetch -f "{hmm_path}" "{names_file}" > /content/pfam/Pfam-A_filtered.hmm
!hmmpress -f /content/pfam/Pfam-A_filtered.hmm

hmm_path = "/content/pfam/Pfam-A_filtered.hmm"

Working...    done.
Indexed 30134 HMMs (30134 names and 30134 accessions).
SSI index written to file /content/pfam/Pfam-A.hmm.ssi
Working...    done.
Pressed and indexed 30133 HMMs (30133 names and 30133 accessions).
Models pressed into binary file:   /content/pfam/Pfam-A_filtered.hmm.h3m
SSI index for binary model file:   /content/pfam/Pfam-A_filtered.hmm.h3i
Profiles (MSV part) pressed into:  /content/pfam/Pfam-A_filtered.hmm.h3f
Profiles (remainder) pressed into: /content/pfam/Pfam-A_filtered.hmm.h3p


In [ ]:
import glob, os
d = f"{DATA_DIR}/raw/shared/interpro_pfam/rolling_2026-10-12"
for f in sorted(glob.glob(d + "/*")):
    print(round(os.path.getsize(f) / 1e6, 1), "Mo", os.path.basename(f))
!head -c 120 "{d}/Pfam-A.hmm" | cat -v | head -3

0.6 Mo Pfam-A.clans.tsv.gz
2246.9 Mo Pfam-A.hmm
0.8 Mo Pfam-A.hmm.dat.gz
418.2 Mo Pfam-A.hmm.gz
531.7 Mo Pfam-A.hmm.h3f
3.0 Mo Pfam-A.hmm.h3i
928.8 Mo Pfam-A.hmm.h3m
1021.4 Mo Pfam-A.hmm.h3p
HMMER3/f [3.3 | Nov 2019]
NAME  03009_C
ACC   PF26733.1


In [ ]:
cpu = os.cpu_count()
protein_fasta = "/content/drive/MyDrive/files/data/raw/zea_mays/ncbi_refseq_zea_b73_v5/Zm-B73-REFERENCE-NAM-5.0_GCF_902167145.1/GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_protein.faa.gz"

cmd = (f'python scan_pfam_domains.py --pfam-hmm "/content/pfam/Pfam-A_filtered.hmm" '
       f'--proteins "{protein_fasta}" '
       f'--species zea_mays --assembly "Zm-B73-REFERENCE-NAM-5.0" '
       f'--out-dir "{DATA_DIR}/pfam_scan/zea_mays" --cpu {cpu} --chunk-size 2000')
print("$", cmd)
!{cmd}

$ python scan_pfam_domains.py --pfam-hmm "/content/pfam/Pfam-A_filtered.hmm" --proteins "/content/drive/MyDrive/files/data/raw/zea_mays/ncbi_refseq_zea_b73_v5/Zm-B73-REFERENCE-NAM-5.0_GCF_902167145.1/GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_protein.faa.gz" --species zea_mays --assembly "Zm-B73-REFERENCE-NAM-5.0" --out-dir "/content/drive/MyDrive/files/data/pfam_scan/zea_mays" --cpu 2 --chunk-size 2000
29 lot(s) deja decoupe(s) dans /content/drive/MyDrive/files/data/pfam_scan/zea_mays/chunks (reutilises, pas re-decoupes).
$ hmmscan sur chunk_00005.faa ...
  termine en 1038 s
$ hmmscan sur chunk_00006.faa ...
  termine en 1076 s
$ hmmscan sur chunk_00023.faa ...
